In [ ]:
# coding: utf-8
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import copy, csv, json, math, random, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import swin_b

UEC        = Path("/workspace/uec256")          # EDIT if different
ARMS       = Path("/workspace/uec256_lt_arms"); ARMS.mkdir(exist_ok=True)
POOL       = Path("/workspace/uec256_synth");   POOL.mkdir(exist_ok=True)
ALIGN_CKPT = "/workspace/align/keep/align_train_20k_seed42.pth"
CACHE      = "/workspace/hf_cache"

DEVICE = "cuda"
SEED   = 42
EPOCHS, BATCH, LR = 30, 32, 1e-4
WARMUP, WD, MIXUP, SMOOTH, EMA_DECAY, TTA_VIEWS, WORKERS = 3, 1e-4, 0.2, 0.1, 0.999, 8, 12

def seed_all(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

free, total = torch.cuda.mem_get_info()
print(f"torch {torch.__version__}   {free/1e9:.1f} GB free of {total/1e9:.1f}")

In [ ]:
!ls /workspace | grep -i uec
!ls /workspace/uec256 2>/dev/null | head
!head -20 /workspace/uec256/category.txt 2>/dev/null
!head -50 /workspace/classify_uec256.py

In [ ]:
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16,
    safety_checker=None, cache_dir=CACHE).to("cuda")
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.set_progress_bar_config(disable=True)

TEST = ["sushi", "rice", "miso soup", "sukiyaki", "oden",
        "chawan-mushi", "goya chanpuru", "nanbanzuke", "kinpira-style sauteed burdock"]

import matplotlib.pyplot as plt
g = torch.Generator("cuda").manual_seed(42)
for name in TEST:
    imgs = pipe([f"A photo of {name}, a type of food."]*3,
                num_inference_steps=50, guidance_scale=2.0, generator=g).images
    fig, ax = plt.subplots(1, 3, figsize=(9, 3))
    for a, im in zip(ax, imgs): a.imshow(im); a.axis("off")
    fig.suptitle(name); plt.show()

In [ ]:
BETTER = [
  ("nanbanzuke", "nanbanzuke, Japanese fried fish marinated in sweet vinegar with sliced onion and chili"),
  ("kinpira burdock", "kinpira gobo, julienned burdock root and carrot stir-fried in soy sauce and sesame"),
  ("chawan-mushi", "chawanmushi, Japanese savory steamed egg custard in a small cup"),
  ("goya chanpuru", "goya champuru, Okinawan stir fry of bitter melon, tofu, egg and pork"),
]
for short, long in BETTER:
    imgs = pipe([f"A photo of {long}."]*3, num_inference_steps=50,
                guidance_scale=2.0, generator=g).images
    fig, ax = plt.subplots(1, 3, figsize=(9,3))
    for a, im in zip(ax, imgs): a.imshow(im); a.axis("off")
    fig.suptitle(short); plt.show()

In [ ]:
!ls /workspace | grep -i uec
!ls /workspace/uec256 2>/dev/null | head -20
!head -5 /workspace/uec256/category.txt 2>/dev/null
!sed -n '1,120p' /workspace/classify_uec256.py

In [ ]:
!ls /root/data/UECFood256/UECFOOD256 2>/dev/null | head -5 || echo "MISSING"
!ls /root/data/UECFood256/UECFOOD256/category.txt 2>/dev/null && head -5 /root/data/UECFood256/UECFOOD256/category.txt

In [ ]:
from collections import defaultdict
from pathlib import Path
import numpy as np

UEC_ROOT = Path("/root/data/UECFood256/UECFOOD256")
cats = sorted([d for d in UEC_ROOT.iterdir() if d.is_dir() and d.name.isdigit()],
              key=lambda d: int(d.name))
counts = []
for d in cats:
    bb = d/"bb_info.txt"
    n = len(bb.read_text().splitlines()) - 1 if bb.exists() else 0
    counts.append(n)
counts = np.array(counts)
print(f"{len(cats)} classes, {counts.sum()} instances")
print(f"min {counts.min()}   max {counts.max()}   median {int(np.median(counts))}   "
      f"imbalance ratio {counts.max()/max(counts.min(),1):.1f}")
print(f"classes under 150: {(counts<150).sum()}   under 120: {(counts<120).sum()}")

names = {}
cf = UEC_ROOT/"category.txt"
if cf.exists():
    for ln in cf.read_text().splitlines()[1:]:
        p = ln.split("\t")
        if len(p) >= 2: names[int(p[0])] = p[1].strip()
    print(f"\n{len(names)} class names loaded")
    order = np.argsort(counts)
    print("\nsmallest 10:")
    for i in order[:10]:
        print(f"  {names.get(int(cats[i].name), '?'):<40} {counts[i]}")
    print("largest 5:")
    for i in order[-5:]:
        print(f"  {names.get(int(cats[i].name), '?'):<40} {counts[i]}")

In [ ]:
!sed -n '120,260p' /workspace/classify_uec256.py